# RDFNet Training + VOC-FOG Evaluation

Trains RDFNet from scratch on VOC-FOG's `train/` split and evaluates the resulting checkpoint on VOC-FOG's `test/` split, using the code from `github.com/habibour/adverse_weather_ibject_detection`.

### Before running
1. In the right sidebar, click **+ Add Data** and attach:
   - `mdhabibourrahman/voc-fog`
2. **Settings > Accelerator > GPU T4 x2** (this notebook launches training with `torchrun` for 2-GPU DDP).
3. Run all cells top to bottom. Training runs the repo's default recipe (Freeze 100 + Unfreeze 300 epochs @ 640x640) -- this can run for many hours and may exceed a single Kaggle session. Checkpoints are saved every 10 epochs under `logs/`; if the session times out, re-attach this notebook's `/kaggle/working` output as an input dataset in a new session and point `model_path` in the config cell at the saved `last_epoch_weights.pth` to continue (not automated here).

In [ ]:
%cd /kaggle/working
!rm -rf adverse_weather_ibject_detection
!git clone --depth 1 https://github.com/habibour/adverse_weather_ibject_detection.git
%cd /kaggle/working/adverse_weather_ibject_detection/RDFNet

In [ ]:
!pip install -q opencv-python-headless tqdm thop
import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
n_gpus = torch.cuda.device_count()
print(f"GPU count: {n_gpus}")
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
if n_gpus < 2:
    print("\nWARNING: this notebook launches training with torchrun --nproc_per_node=2 (DDP).\n"
          "Go to Settings -> Accelerator -> GPU T4 x2 and restart the kernel for a 2-GPU session.")

## Inspect the attached dataset

`voc-fog` has separate `train/` and `test/` folders rather than a single `VOC2007/` tree. This cell prints the tree so we can confirm the exact subfolder names before building annotation files.

In [ ]:
import os

VOC_FOG_BASE = "/kaggle/input/datasets/mdhabibourrahman/voc-fog/VOC_FOG"

def print_tree(root, max_depth=4, max_files_per_dir=10):
    root = root.rstrip('/')
    base_depth = root.count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        depth = dirpath.count(os.sep) - base_depth
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = '  ' * depth
        print(f"{indent}{os.path.basename(dirpath)}/")
        for f in sorted(filenames)[:max_files_per_dir]:
            print(f"{indent}  {f}")
        if len(filenames) > max_files_per_dir:
            print(f"{indent}  ... ({len(filenames) - max_files_per_dir} more files)")

print(f"===== voc-fog: {VOC_FOG_BASE} =====")
if os.path.exists(VOC_FOG_BASE):
    print_tree(VOC_FOG_BASE)
else:
    print("NOT FOUND -- check the exact path under /kaggle/input")

## Resolve train/ and test/ layout

- **train/**: `Annotations/` + `JPEGImages/` (clean) + `VOC2007-FOG/` (foggy) -- used to build the paired fog/clean training lists.
- **test/**: `Annotations/` + `VOCtest-FOG/` -- used both as the periodic validation set during training and as the final held-out test set.

Resolved by searching for the directory that actually contains the expected subfolders, rather than assuming a fixed nesting depth (this dataset's layout has shifted between uploads before).

In [ ]:
def find_data_root(base, required_subdirs):
    for dirpath, dirnames, _ in os.walk(base):
        if all(d in dirnames for d in required_subdirs):
            return dirpath
    raise FileNotFoundError(f"No directory under {base} contains all of {required_subdirs}")

TRAIN_ROOT = find_data_root(VOC_FOG_BASE, ("Annotations", "JPEGImages", "VOC2007-FOG"))
TRAIN_FOG_DIR   = os.path.join(TRAIN_ROOT, "VOC2007-FOG")
TRAIN_CLEAN_DIR = os.path.join(TRAIN_ROOT, "JPEGImages")
TRAIN_ANN_DIR   = os.path.join(TRAIN_ROOT, "Annotations")

TEST_ROOT = find_data_root(VOC_FOG_BASE, ("Annotations", "VOCtest-FOG"))
TEST_FOG_DIR = os.path.join(TEST_ROOT, "VOCtest-FOG")
TEST_ANN_DIR = os.path.join(TEST_ROOT, "Annotations")

print(f"train: fog={TRAIN_FOG_DIR}\n       clean={TRAIN_CLEAN_DIR}\n       ann={TRAIN_ANN_DIR}")
print(f"test : fog={TEST_FOG_DIR}\n       ann={TEST_ANN_DIR}")

## Build annotation files

`train.py` reads three files: `train_annotation_path` (foggy) and `clear_annotation_path` (clean) which must be index-aligned pairs of the same underlying photo, plus `val_annotation_path` (foggy test images, used by the training loop's periodic `EvalCallback` and reused below as the final test set).

Foggy filenames may carry a `_f<N>` fog-variant suffix over the clean image's id (e.g. `2007_000027_f0.jpg` -> clean id `2007_000027`); this is stripped to look up the matching clean image and XML annotation. Any image that fails to decode (PIL) or has an unparseable/empty-box annotation is skipped and counted, instead of crashing the run -- this dataset has had corrupt files before.

In [ ]:
import re
import xml.etree.ElementTree as ET
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

CLASSES_PATH = 'model_data/rtts_classes.txt'
with open(CLASSES_PATH) as f:
    CLASSES = [c.strip() for c in f if c.strip()]
print(f"Classes ({len(CLASSES)}): {CLASSES}")

FOG_SUFFIX_RE = re.compile(r'^(?P<base>.+?)_f\d+$')

def base_stem(stem):
    m = FOG_SUFFIX_RE.match(stem)
    return m.group('base') if m else stem

def is_valid_image(path):
    try:
        with Image.open(path) as im:
            im.load()
        return True
    except (UnidentifiedImageError, OSError):
        return False

def parse_boxes(xml_path):
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        return None
    boxes = []
    for obj in root.iter('object'):
        diff = obj.find('difficult')
        if diff is not None and int(diff.text) == 1:
            continue
        cls = obj.find('name').text.strip()
        if cls not in CLASSES:
            continue
        bb = obj.find('bndbox')
        coords = [int(float(bb.find(t).text)) for t in ('xmin', 'ymin', 'xmax', 'ymax')]
        boxes.append(','.join(map(str, coords)) + ',' + str(CLASSES.index(cls)))
    return boxes

def box_suffix(boxes):
    return ''.join(' ' + b for b in boxes)

def build_train_pairs(fog_dir, clean_dir, ann_dir, out_fog_path, out_clean_path):
    fog_lines, clean_lines = [], []
    skipped = {'no_clean': 0, 'no_ann': 0, 'no_boxes': 0, 'bad_fog_img': 0, 'bad_clean_img': 0}
    fog_files = sorted(f for f in os.listdir(fog_dir) if f.lower().endswith('.jpg'))
    for fname in tqdm(fog_files, desc='pairing train fog/clean'):
        fog_stem = os.path.splitext(fname)[0]
        stem = base_stem(fog_stem)
        clean_path = os.path.join(clean_dir, stem + '.jpg')
        ann_path = os.path.join(ann_dir, stem + '.xml')
        fog_path = os.path.join(fog_dir, fname)
        if not os.path.exists(clean_path):
            skipped['no_clean'] += 1
            continue
        if not os.path.exists(ann_path):
            skipped['no_ann'] += 1
            continue
        boxes = parse_boxes(ann_path)
        if not boxes:
            skipped['no_boxes'] += 1
            continue
        if not is_valid_image(fog_path):
            skipped['bad_fog_img'] += 1
            continue
        if not is_valid_image(clean_path):
            skipped['bad_clean_img'] += 1
            continue
        suffix = box_suffix(boxes)
        fog_lines.append(fog_path + suffix)
        clean_lines.append(clean_path + suffix)
    with open(out_fog_path, 'w') as f:
        f.write('\n'.join(fog_lines) + '\n')
    with open(out_clean_path, 'w') as f:
        f.write('\n'.join(clean_lines) + '\n')
    print(f"train: {len(fog_lines)} pairs written -> {out_fog_path}, {out_clean_path} | skipped: {skipped}")
    return len(fog_lines)

def build_val(fog_dir, ann_dir, out_path):
    lines, ids = [], []
    skipped = {'no_ann': 0, 'no_boxes': 0, 'bad_img': 0}
    fog_files = sorted(f for f in os.listdir(fog_dir) if f.lower().endswith('.jpg'))
    for fname in tqdm(fog_files, desc='building val/test list'):
        fog_stem = os.path.splitext(fname)[0]
        stem = base_stem(fog_stem)
        ann_path = os.path.join(ann_dir, stem + '.xml')
        fog_path = os.path.join(fog_dir, fname)
        if not os.path.exists(ann_path):
            skipped['no_ann'] += 1
            continue
        boxes = parse_boxes(ann_path)
        if not boxes:
            skipped['no_boxes'] += 1
            continue
        if not is_valid_image(fog_path):
            skipped['bad_img'] += 1
            continue
        lines.append(fog_path + box_suffix(boxes))
        ids.append(fog_stem)
    with open(out_path, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    print(f"val/test: {len(lines)} images written -> {out_path} | skipped: {skipped}")
    return ids

NUM_TRAIN_PAIRS = build_train_pairs(TRAIN_FOG_DIR, TRAIN_CLEAN_DIR, TRAIN_ANN_DIR,
                                     '2007_train_fog.txt', '2007_train.txt')
TEST_IDS = build_val(TEST_FOG_DIR, TEST_ANN_DIR, '2007_val_fog.txt')

## Configure training

Repo default recipe (Freeze 100 + Unfreeze 300 epochs, 640x640, DDP across 2 GPUs).

In [ ]:
import sys

config_content = """import os

Cuda            = True
seed            = 114514
distributed     = True
sync_bn         = False
fp16            = False

classes_path    = 'model_data/rtts_classes.txt'
anchors_path    = 'model_data/yolo_anchors.txt'
anchors_mask    = [[6, 7, 8], [3, 4, 5], [0, 1, 2]]
model_path      = 'model_data/yolov7_tiny_weights.pth'
input_shape     = [640, 640]

pretrained      = False
Init_Epoch          = 0
Freeze_Epoch        = 100
Freeze_batch_size   = 16
UnFreeze_Epoch      = 300
Unfreeze_batch_size = 16
Freeze_Train        = True
Init_lr             = 1e-2
Min_lr              = Init_lr * 0.01
optimizer_type      = \"sgd\"
momentum            = 0.937
weight_decay        = 5e-4
lr_decay_type       = \"cos\"
save_period         = 10
save_dir            = 'logs'
eval_flag           = True
eval_period         = 10
num_workers         = 2

train_annotation_path = '2007_train_fog.txt'
val_annotation_path   = '2007_val_fog.txt'
clear_annotation_path = '2007_train.txt'
"""

with open('config.py', 'w') as f:
    f.write(config_content)

for k in list(sys.modules.keys()):
    if k == 'config':
        del sys.modules[k]
import config as cfg
print('config.py written:')
print(f'  distributed    = {cfg.distributed}')
print(f'  fp16           = {cfg.fp16}  (kept off -- utils_fit.py\'s fp16 branch is broken upstream: it')
print(f'                     forwards only the hazy batch through a backbone that unconditionally splits')
print(f'                     a hazy+clean concatenated batch, and never computes loss_dehazy -- see')
print(f'                     utils/utils_fit.py lines 38-45 vs 30-37)')
print(f'  input_shape    = {cfg.input_shape}')
print(f'  Freeze_Epoch   = {cfg.Freeze_Epoch}')
print(f'  UnFreeze_Epoch = {cfg.UnFreeze_Epoch}')
print(f'  train pairs    = {NUM_TRAIN_PAIRS}')
print(f'  val/test images= {len(TEST_IDS)}')

## Train

Launches `train.py` under `torchrun` for DDP. Progress (loss, periodic VOC-FOG-test mAP every `eval_period` epochs) prints below; checkpoints land in `logs/loss_<timestamp>/`.

In [ ]:
import time

t_start = time.time()
ret = os.system(f'torchrun --nproc_per_node={max(torch.cuda.device_count(), 1)} --master_port=12355 train.py')
elapsed = (time.time() - t_start) / 3600
print(f"\nTraining exit code: {ret} | elapsed: {elapsed:.2f}h")
if ret != 0:
    print("Training did not exit cleanly -- check the log above for the traceback before proceeding.")

## Final evaluation on the VOC-FOG test set

Same VOC-style mAP@0.5 logic as `get_map.py` / `rdfnet_baseline_eval.ipynb`'s `evaluate()`, pointed at the best checkpoint from this training run.

In [ ]:
import glob

weight_candidates = sorted(glob.glob('logs/*/best_epoch_weights.pth')) or sorted(glob.glob('logs/*/last_epoch_weights.pth'))
if not weight_candidates:
    raise FileNotFoundError("No checkpoint found under logs/*/ -- did training complete at least one epoch?")
WEIGHT_PATH = weight_candidates[-1]
print('Using weights:', WEIGHT_PATH)

In [ ]:
from utils.utils import get_classes
from utils.utils_map import get_map
from yolo import YOLO

ANCHORS_PATH = 'model_data/yolo_anchors.txt'

def evaluate(dataname, image_ids, images_dir, annotations_dir, model_path, image_ext='.jpg',
             classes_path=CLASSES_PATH, anchors_path=ANCHORS_PATH,
             min_overlap=0.5, confidence=0.001, nms_iou=0.5, score_threshold=0.5):
    map_out_path = f'/kaggle/working/map_out-{dataname}'
    for sub in ['ground-truth', 'detection-results', 'images-optional']:
        os.makedirs(os.path.join(map_out_path, sub), exist_ok=True)

    class_names, _ = get_classes(classes_path)
    print(f"[{dataname}] {len(image_ids)} images | loading model from {model_path}")
    yolo = YOLO(model_path=model_path, classes_path=classes_path, anchors_path=anchors_path,
                confidence=confidence, nms_iou=nms_iou, cuda=torch.cuda.is_available())

    skipped = []
    for image_id in tqdm(image_ids, desc=f'[{dataname}] predicting'):
        image_path = os.path.join(images_dir, image_id + image_ext)
        try:
            image = Image.open(image_path)
            image.load()
        except (UnidentifiedImageError, OSError) as e:
            skipped.append((image_id, str(e)))
            open(os.path.join(map_out_path, f"detection-results/{image_id}.txt"), "w").close()
            continue
        yolo.get_map_txt(image_id, image, class_names, map_out_path)

    if skipped:
        print(f"[{dataname}] WARNING: skipped {len(skipped)}/{len(image_ids)} unreadable image(s) "
              f"(treated as zero detections):")
        for image_id, err in skipped[:20]:
            print(f"    {image_id}{image_ext}: {err}")

    bad_xml = []
    for image_id in tqdm(image_ids, desc=f'[{dataname}] ground truth'):
        with open(os.path.join(map_out_path, f"ground-truth/{image_id}.txt"), "w") as new_f:
            try:
                root = ET.parse(os.path.join(annotations_dir, f"{base_stem(image_id)}.xml")).getroot()
            except ET.ParseError as e:
                bad_xml.append((image_id, str(e)))
                continue
            for obj in root.findall('object'):
                difficult_flag = obj.find('difficult') is not None and int(obj.find('difficult').text) == 1
                obj_name = obj.find('name').text
                if obj_name not in class_names:
                    continue
                bnd = obj.find('bndbox')
                left, top = bnd.find('xmin').text, bnd.find('ymin').text
                right, bottom = bnd.find('xmax').text, bnd.find('ymax').text
                suffix = " difficult" if difficult_flag else ""
                new_f.write(f"{obj_name} {left} {top} {right} {bottom}{suffix}\n")

    if bad_xml:
        print(f"[{dataname}] WARNING: {len(bad_xml)} unparseable annotation(s) (treated as 0 objects):")
        for image_id, err in bad_xml[:20]:
            print(f"    {image_id}.xml: {err}")

    print(f"[{dataname}] computing mAP")
    mAP = get_map(min_overlap, True, score_threhold=score_threshold, path=map_out_path)
    return mAP

test_mAP = evaluate('VOC-FOG-test', TEST_IDS, TEST_FOG_DIR, TEST_ANN_DIR, WEIGHT_PATH, image_ext='.jpg')
print(f"\n=== Final VOC-FOG test mAP@0.5: {test_mAP:.2f}% ===")

## Save outputs

In [ ]:
import shutil

out_dir = '/kaggle/working/rdfnet_train_results'
os.makedirs(out_dir, exist_ok=True)

for pattern in ['logs/*/best_epoch_weights.pth', 'logs/*/last_epoch_weights.pth',
                'logs/*/epoch_map.txt', 'logs/*/epoch_map.png', 'logs/*/epoch_loss.png']:
    for f in glob.glob(pattern):
        dest = os.path.join(out_dir, os.path.basename(os.path.dirname(f)) + '_' + os.path.basename(f))
        shutil.copy(f, dest)
        print('Saved:', dest)

with open(os.path.join(out_dir, 'final_test_map.txt'), 'w') as f:
    f.write(f'VOC-FOG test mAP@0.5: {test_mAP:.2f}%\n')

print(f"\nDone. Download from Kaggle Output tab -> {os.path.basename(out_dir)}/")